# Documentation: Message Types in LangChain

LLMs process information as a sequence of messages. Understanding the different message types is essential for maintaining chat history, providing instructions, and handling tool interactions.

#### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.
Messages are objects that contain:
 - Role - Identifies the message type (e.g. system, user)
 - Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
 - Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

---

## 1. Core Message Types

### **SystemMessage**
* **Role:** The "Internal Instructions."
* **Purpose:** Sets the behavior, persona, and constraints for the AI. It is usually sent at the very beginning of a conversation.
* **Example:** "You are a senior financial analyst. Be concise and use professional terminology."

### **HumanMessage**
* **Role:** The "User."
* **Purpose:** Represents the input from the person interacting with the model.
* **Example:** "What is the current stock price of NVIDIA?"

### **AIMessage**
* **Role:** The "Assistant."
* **Purpose:** Represents the response generated by the LLM. 
* **Note:** When building a chatbot, you must save these messages back into your memory/history so the model remembers what it said previously.

---

## 2. Tool-Related Message Types

When working with agents and tools, two additional message types are used to handle the "Request -> Execution -> Result" loop.

### **AIMessage (with Tool Calls)**
When a model decides to use a tool, it returns an `AIMessage` that contains a `tool_calls` field instead of (or in addition to) text content. This is the model saying: *"I am pausing my response to ask the system to run this function."*

### **ToolMessage**
* **Role:** The "Tool Output."
* **Purpose:** This message contains the **result** of a tool's execution. It must include a `tool_call_id` that matches the ID from the `AIMessage` so the model knows which result belongs to which request.

---

## 3. The Message Flow Diagram



---

## 4. Comparison Table

| Message Type | Sender | Typical Content | Purpose |
| :--- | :--- | :--- | :--- |
| **System** | Developer | Instructions/Persona | Set context and rules. |
| **Human** | User | Questions/Commands | Provide the prompt. |
| **AI** | LLM | Text or Tool Calls | Respond to the user. |
| **Tool** | System/Code | Function Results | Provide data back to the LLM. |

---

## 5. Best Practices

*  `System Message Stability` : Keep the SystemMessage at the index 0 of your list. Moving it or removing it can cause the model to "forget" its instructions.
* `Trim History`: As conversations get long, the message list consumes more tokens. Use a "Trimmer" to keep only the most recent messages while preserving the SystemMessage.
* `Content as Lists`: Modern models (like GPT-4o or Claude 3.5) allow content to be a list of dictionaries for multi-modal inputs (e.g., text + images).

## 6. Code Implementation Example


In [1]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# A typical conversation history
messages = [
    SystemMessage(content="You are a helpful travel agent."),
    HumanMessage(content="I want to go to Paris."),
    AIMessage(content="That sounds lovely! When are you planning to visit?")
]

# Adding a Tool interaction
# 1. Model asks to use a tool
tool_request = AIMessage(
    content="",
    tool_calls=[{"name": "get_flights", "args": {"destination": "Paris"}, "id": "call_123"}]
)

# 2. System provides the tool result
tool_response = ToolMessage(
    content="Found 3 flights starting at $500.",
    tool_call_id="call_123"
)

messages.extend([tool_request, tool_response])
messages

[SystemMessage(content='You are a helpful travel agent.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I want to go to Paris.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='That sounds lovely! When are you planning to visit?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'get_flights', 'args': {'destination': 'Paris'}, 'id': 'call_123', 'type': 'tool_call'}], invalid_tool_calls=[]),
 ToolMessage(content='Found 3 flights starting at $500.', tool_call_id='call_123')]

In [1]:
from langchain.chat_models import init_chat_model
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash")
model 


ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x00000242717F4BF0>, default_metadata=(), model_kwargs={})

In [3]:
model.invoke("Please tell me about AI Messages")

AIMessage(content='"AI Messages" is a broad term that generally refers to any form of communication where Artificial Intelligence plays a significant role, either in generating the message, processing it, or enhancing its delivery and understanding.\n\nLet\'s break down what "AI Messages" can encompass:\n\n1.  **AI-Generated Messages (Output):** These are messages created directly by AI models.\n    *   **Chatbot Responses:** When you interact with a customer service bot, a virtual assistant (like Siri, Alexa, Google Assistant), or a generative AI model (like ChatGPT, Bard, Claude), the text (or sometimes voice) responses you receive are AI-generated messages.\n    *   **Content Creation:** AI can write articles, marketing copy, social media posts, emails, product descriptions, creative stories, poems, and even code snippets.\n    *   **Summaries:** AI can condense long documents, emails, or conversations into concise summaries.\n    *   **Personalized Communications:** AI can generate


Message types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

In [6]:
system_msg = SystemMessage("You are a helpful coding assistant.")
messages = [
    system_msg,
    HumanMessage("How do I create a Rest API")
]
response = model.invoke(messages)
print(response.content)


Creating a REST API is a fundamental skill for modern web development, allowing different applications to communicate with each other. This guide will walk you through the core concepts and provide a practical example using **Python with Flask**, a popular and lightweight web framework, and **SQLite** for the database.

---

## What is a REST API?

**REST** (Representational State Transfer) is an architectural style for designing networked applications. A **REST API** (Application Programming Interface) is a set of rules that allows different software applications to communicate with each other over the internet.

Key principles of REST:

1.  **Client-Server Architecture:** The client (e.g., a web browser, mobile app) is separate from the server (where the API resides).
2.  **Statelessness:** Each request from the client to the server must contain all the information needed to understand the request. The server should not store any client context between requests.
3.  **Cacheability:**

In [7]:
## Detailed info to the LLM through System message
system_msg = SystemMessage(""" 
Yor are a senior python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in explanations. 
""")
messages = [
    system_msg,
    HumanMessage("How do I create a Rest API")
]
response = model.invoke(messages)
print(response.content)


Creating a REST API involves defining a set of endpoints (URLs) that clients can interact with using standard HTTP methods (GET, POST, PUT, DELETE) to perform operations on resources. The responses are typically in JSON format.

In Python, popular choices for building REST APIs include:

1.  **FastAPI**: Modern, fast (high performance), and built on standard Python type hints. It automatically generates API documentation (Swagger UI, ReDoc).
2.  **Flask**: A lightweight micro-framework that provides flexibility. You'll often use extensions like Flask-RESTful or Flask-RESTX for API development.
3.  **Django REST Framework (DRF)**: A powerful and flexible toolkit for building Web APIs on top of Django. Ideal for applications already using Django's ORM and features.

I'll demonstrate using **FastAPI** as it's a modern, high-performance choice that inherently supports many API best practices like data validation and automatic documentation.

---

### Step-by-Step Guide with FastAPI

**1. P

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
from langchain.chat_models import init_chat_model
model=init_chat_model("gpt-4.1")
model

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000024212778B60>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000024212C61760>, root_client=<openai.OpenAI object at 0x0000024210FAB170>, root_async_client=<openai.AsyncOpenAI object at 0x0000024212778AD0>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [4]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

2 + 2 equals 4.


In [5]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')